In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
symlist = self.get_updated_symbol_list(age_ub=12000)

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_earning = self.count_days_from_earning_reports(df_quotes)
if df_earning.shape[0] == 0:
    d2e = {}
else:
    d2e = df_earning['earningDays'].to_dict()
print('Days to E:', d2e)
df_raw = self.build_option_df(symlist)
px.bar(self.check_data_age(df_raw), barmode='group', width=60*len(symlist), height=300).show()
df_put = self.select_options_by_type(df_raw, 'put')

In [ ]:
df_call = self.select_options_by_type(df_raw, 'call')
_df = self.calc_spread_stats(df_call)
px.bar(_df.sort_values(by='mean spread'), barmode='group', width=60*len(symlist))

### Call Options: ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
dfc = self.compute_all_time_decay_metrics_for_symbols(df_call, symlist, d2e, ignore_no_bid=True, exclude_0dte=True, oi_lb=100)
print('hdte_resid check:', dfc[(dfc.hdte_resid - dfc.resid) <= -1e-6].shape)

In [ ]:
hdte_resid_lb = 0.9
overpaid_ub = 0.05
spread_ub = 5
_filter = (dfc.dte >= 90) & (dfc.hdte_resid >= hdte_resid_lb) & (dfc.overpaid <= overpaid_ub) & (dfc.pctSpread <= spread_ub)
_filter = _filter & (dfc.symbol != 'TLT')
_dfc = dfc[_filter].drop(columns=['dth', 'dtz', 'dthr', 'dtzr']).sort_values(by='leverage', ascending=False)
_dfc.head(20)

In [ ]:
_filter = (dfc.pctSpread <= 10) & (dfc.strike <= dfc.lastPrice) & (dfc.dte >= 60)
px.scatter(dfc[_filter].sort_values(by='leverage', ascending=False).head(1000), x='hdte_resid', y='leverage', color='symbol', height=600)

In [ ]:
dfc[(dfc.symbol=='TSM') & (dfc.dte >= 90) & (dfc.strike <= dfc.lastPrice) & (dfc.leverage <= 8)].sort_values(by='leverage', ascending=False).head(20)

### The End